# Color Segmentation Pipeline — SAM + ControlNet img2img

**Pipeline:**
1. Sample random color hint strokes from GT images
2. Use already-generated ControlNet images as segmentation input
3. Use prompted SAM for the outer garment mask and automatic SAM for internal regions
4. Assign user hint colors to regions selected by hints
5. LAB-blend selected colored regions with the ControlNet structure image
6. Run **ControlNet img2img** with the blended image as init and the original sketch as control

**Input:** ControlNet-generated images + matching sketches + GT images for hint sampling

**Output:** Region-colored init images and final ControlNet-img2img outputs


In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q',
    'diffusers==0.21.4',
    'huggingface_hub==0.23.4',
    'transformers==4.38.2',
    'accelerate==0.27.2',
], check=True)

print('Done — restart kernel now before running any other cells')

In [1]:
# ── Install ────────────────────────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q',
    'diffusers==0.27.2',
    'transformers',
    'accelerate',
    'safetensors',
    'opencv-python',
    'scikit-image',
    'scipy',
], check=True)

# Install SAM
subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/facebookresearch/segment-anything.git'
], check=True)

# Download SAM checkpoint (ViT-B is fastest, good enough)
import os
sam_ckpt = '/kaggle/working/sam_vit_b.pth'
if not os.path.exists(sam_ckpt):
    subprocess.run([
        'wget', '-q', '-O', sam_ckpt,
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
    ], check=True)
    print('SAM checkpoint downloaded.')
else:
    print('SAM checkpoint already exists.')

SAM checkpoint already exists.


In [ ]:
import os, json, glob, random
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from pathlib import Path
from scipy.ndimage import gaussian_filter, binary_closing, binary_fill_holes
from skimage.color import rgb2lab, lab2rgb

from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetImg2ImgPipeline,
    UniPCMultistepScheduler,
)
from segment_anything import (
    SamAutomaticMaskGenerator,
    SamPredictor,
    sam_model_registry,
)

# ── CONFIG ─────────────────────────────────────────────────────────────────
CONFIG = {
    # Paths
    'controlnet_images_dir': '/kaggle/input/datasets/humnafaisal1/controlnet-data/kaggle/working/controlnet_eval_outputs/generated',
    'gt_dir':                '/kaggle/input/datasets/humnafaisal1/controlnet-data/kaggle/working/controlnet_eval_outputs/ground_truth',
    'sketch_dir':            '/kaggle/input/datasets/humnafaisal1/controlnet-data/kaggle/working/controlnet_eval_outputs/sketches',
    'output_dir':            '/kaggle/working/sam_outputs',

    # SAM
    'sam_checkpoint':        '/kaggle/working/sam_vit_b.pth',
    'sam_model_type':        'vit_b',
    'sam_points_per_side':   16,
    'sam_min_mask_area':     800,
    'sam_iou_thresh':        0.88,
    'sam_stability_thresh':  0.92,

    # Color hint generation
    'n_strokes':             6,
    'stroke_min_length':     30,
    'stroke_max_length':     100,
    'stroke_min_width':      4,
    'stroke_max_width':      12,

    # ControlNet img2img
    'sd_model':              'runwayml/stable-diffusion-v1-5',
    'controlnet_model':      'lllyasviel/control_v11p_sd15_lineart',
    # Optional local raw state_dict for your fine-tuned sketch ControlNet.
    'controlnet_state_dict': None,
    'img2img_strength':      0.50,
    'guidance_scale':        7.0,
    'controlnet_scale':      1.5,
    'num_inference_steps':   25,
    'image_size':            512,
    'prompt':                'a single clothing item, centered, studio product photo, light grey background, no model, no mannequin, high quality',
    'negative_prompt':       'person, model, mannequin, hanger, rack, room, floor, wall, props, accessories, text, logo, watermark, extra garment, multiple garments, clutter, colored background, pattern',

    'seed': 42,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## Step 1 — Color Hint Generation

Sample random dragged strokes from GT images.
These simulate what a user would draw on the canvas.

In [ ]:
def sample_color_hints(gt_image_np, n_strokes=6,
                        min_length=30, max_length=100,
                        min_width=4,  max_width=12,
                        img_size=512):
    """
    Sample random dragged color strokes from GT image.
    Strokes are placed only on non-background (garment) pixels.

    Returns:
        color_map : (H,W,3) float32 [0,1] — RGB color at hint pixels
        hint_mask : (H,W,1) float32 [0,1] — 1 where hint exists
    """
    color_map = np.zeros((img_size, img_size, 3), dtype=np.float32)
    hint_mask = np.zeros((img_size, img_size, 1), dtype=np.float32)

    # Find non-background pixels (not near-white)
    non_bg = np.argwhere(np.any(gt_image_np < 0.80, axis=-1))
    if len(non_bg) < 20:
        return color_map, hint_mask   # blank/white image — skip

    n = np.random.randint(max(1, n_strokes-2), n_strokes+3)

    for _ in range(n):
        # Random start on garment
        start  = non_bg[np.random.choice(len(non_bg))]
        y0, x0 = int(start[0]), int(start[1])
        color  = gt_image_np[y0, x0].copy()

        # Random direction, length, width
        angle  = np.random.uniform(0, 2 * np.pi)
        length = np.random.randint(min_length, max_length)
        width  = np.random.randint(min_width, max_width)

        x1 = int(np.clip(x0 + length * np.cos(angle), 0, img_size-1))
        y1 = int(np.clip(y0 + length * np.sin(angle), 0, img_size-1))

        # Rasterize stroke
        n_pts = max(length * 2, 10)
        xs    = np.linspace(x0, x1, n_pts).astype(int)
        ys    = np.linspace(y0, y1, n_pts).astype(int)
        yy, xx = np.ogrid[:img_size, :img_size]
        half_w = width // 2

        for px, py in zip(xs, ys):
            mask_circle = (xx - px)**2 + (yy - py)**2 <= half_w**2
            color_map[mask_circle]    = color
            hint_mask[mask_circle, 0] = 1.0

    return color_map, hint_mask


def visualize_hints(gt_np, color_map, hint_mask, title='Color Hints'):
    """Show GT image alongside the sampled color hints."""
    hint_vis = gt_np.copy()
    hint_vis[hint_mask[:,:,0] > 0] = color_map[hint_mask[:,:,0] > 0]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(gt_np);   axes[0].set_title('GT Image');    axes[0].axis('off')
    axes[1].imshow(color_map); axes[1].set_title('Color Map'); axes[1].axis('off')
    axes[2].imshow(hint_vis); axes[2].set_title('Hints on GT'); axes[2].axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# ── Test on a few samples ──────────────────────────────────────────────────
gt_paths = sorted(Path(CONFIG['gt_dir']).glob('*.*'))[:5]
print(f'Testing hint generation on {len(gt_paths)} GT images...')

for gt_path in gt_paths[:3]:
    gt_pil = Image.open(gt_path).convert('RGB').resize(
        (CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)
    gt_np  = np.array(gt_pil, dtype=np.float32) / 255.0

    color_map, hint_mask = sample_color_hints(
        gt_np,
        n_strokes  = CONFIG['n_strokes'],
        min_length = CONFIG['stroke_min_length'],
        max_length = CONFIG['stroke_max_length'],
        min_width  = CONFIG['stroke_min_width'],
        max_width  = CONFIG['stroke_max_width'],
        img_size   = CONFIG['image_size'],
    )
    print(f'  {gt_path.name}: {int(hint_mask.sum())} hint pixels')
    visualize_hints(gt_np, color_map, hint_mask, title=gt_path.name)

## Step 2 — Load SAM and Segment ControlNet Images

In [ ]:
print('Loading SAM...')
sam = sam_model_registry[CONFIG['sam_model_type']](
    checkpoint=CONFIG['sam_checkpoint']
).to(device)
sam.eval()

mask_generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side         = CONFIG['sam_points_per_side'],
    pred_iou_thresh         = CONFIG['sam_iou_thresh'],
    stability_score_thresh  = CONFIG['sam_stability_thresh'],
    min_mask_region_area    = CONFIG['sam_min_mask_area'],
)
predictor = SamPredictor(sam)
print('SAM loaded.')


def _sketch_foreground_point(sketch_np):
    """Pick a positive SAM point from the center of the sketch line bounding box."""
    line = sketch_np < 0.6
    ys, xs = np.where(line)
    if len(xs) < 10:
        H, W = sketch_np.shape[:2]
        return np.array([W // 2, H // 2])
    return np.array([int(np.median(xs)), int(np.median(ys))])


def prompted_outer_mask(image_np_uint8, sketch_np=None):
    """Prompt SAM with garment foreground + corner background points."""
    H, W = image_np_uint8.shape[:2]
    predictor.set_image(image_np_uint8)

    if sketch_np is not None:
        fg = _sketch_foreground_point(sketch_np)
    else:
        img = image_np_uint8.astype(np.float32) / 255.0
        non_white = ~((img[:,:,0] > 0.88) & (img[:,:,1] > 0.88) & (img[:,:,2] > 0.88))
        ys, xs = np.where(non_white)
        fg = np.array([int(np.median(xs)), int(np.median(ys))]) if len(xs) else np.array([W//2, H//2])

    point_coords = np.array([
        fg,
        [5, 5], [W - 6, 5], [5, H - 6], [W - 6, H - 6]
    ], dtype=np.float32)
    point_labels = np.array([1, 0, 0, 0, 0], dtype=np.int32)

    masks, scores, _ = predictor.predict(
        point_coords=point_coords,
        point_labels=point_labels,
        multimask_output=True,
    )

    areas = masks.reshape(masks.shape[0], -1).sum(axis=1)
    valid = (areas > 0.01 * H * W) & (areas < 0.95 * H * W)
    if valid.any():
        idxs = np.where(valid)[0]
        best = idxs[np.argmax(scores[idxs])]
    else:
        best = int(np.argmax(scores))

    outer = masks[best].astype(bool)
    outer = binary_closing(outer, structure=np.ones((5,5), dtype=bool))
    outer = binary_fill_holes(outer)
    return outer


def segment_image(image_np_uint8):
    """Segment a ControlNet-generated image into SAM regions."""
    masks = mask_generator.generate(image_np_uint8)
    return sorted(masks, key=lambda m: m['area'], reverse=True)


def filter_masks_to_outer(masks, outer_mask, min_overlap=0.5):
    """Keep only automatic SAM masks mostly inside the prompted garment mask."""
    filtered = []
    for m in masks:
        seg = m['segmentation'].astype(bool)
        area = seg.sum()
        if area == 0:
            continue
        overlap = (seg & outer_mask).sum() / area
        if overlap >= min_overlap:
            m = dict(m)
            m['segmentation'] = seg & outer_mask
            m['area'] = int(m['segmentation'].sum())
            if m['area'] >= CONFIG['sam_min_mask_area']:
                filtered.append(m)
    return sorted(filtered, key=lambda m: m['area'])


def visualize_masks(image_np, outer_mask, masks, title='SAM Segmentation'):
    """Overlay outer mask and internal masks on the image."""
    overlay = image_np.copy().astype(float) / 255.0
    colors  = plt.cm.tab20(np.linspace(0, 1, max(len(masks), 1)))

    overlay[outer_mask] = 0.65 * overlay[outer_mask] + 0.35 * np.array([0.0, 1.0, 0.0])
    for i, m in enumerate(masks):
        seg = m['segmentation']
        color = colors[i % len(colors)][:3]
        overlay[seg] = 0.5 * overlay[seg] + 0.5 * np.array(color)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(image_np); axes[0].set_title('ControlNet Output'); axes[0].axis('off')
    axes[1].imshow(outer_mask, cmap='gray'); axes[1].set_title('Prompted Outer Mask'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title(f'Internal masks: {len(masks)}'); axes[2].axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# ── Test SAM on one ControlNet image ──────────────────────────────────────
cn_images = sorted(glob.glob(os.path.join(CONFIG['controlnet_images_dir'], '*.png')))
if len(cn_images) == 0:
    raise FileNotFoundError(f"No ControlNet images found in {CONFIG['controlnet_images_dir']}")

print(f'Found {len(cn_images)} ControlNet images')
test_img = Image.open(cn_images[0]).convert('RGB').resize((CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)
test_np = np.array(test_img)
outer_test = prompted_outer_mask(test_np)
test_masks = filter_masks_to_outer(segment_image(test_np), outer_test)
print(f'SAM found {len(test_masks)} internal garment regions')
visualize_masks(test_np, outer_test, test_masks)


## Step 3 — Assign Colors to Regions

Match each segmented region to the nearest/most-covered color hint.
Build a colored region map for img2img input.

In [ ]:
def assign_colors_to_masks(masks, color_map, hint_mask, image_np,
                           outer_mask=None, fill_unhinted_with_nearest=False):
    """
    Assign hint colors to SAM regions. By default, only masks containing hints
    are recolored; unhinted regions keep the ControlNet image color.
    """
    H, W = hint_mask.shape[:2]
    img_float = image_np.astype(float) / 255.0
    colored_np = img_float.copy()
    filled_mask = np.zeros((H, W), dtype=bool)

    if outer_mask is None:
        outer_mask = np.ones((H, W), dtype=bool)

    hint_points = np.argwhere(hint_mask[:,:,0] > 0.05)
    has_hints = len(hint_points) > 0
    region_colors = []

    if not masks and outer_mask is not None and outer_mask.any():
        masks = [{'segmentation': outer_mask, 'area': int(outer_mask.sum())}]

    for m in masks:
        seg = m['segmentation'].astype(bool) & outer_mask
        if seg.sum() == 0 or not has_hints:
            continue

        region_hint_mask = seg & (hint_mask[:,:,0] > 0.05)
        if region_hint_mask.any():
            region_color = np.median(color_map[region_hint_mask], axis=0)
        elif fill_unhinted_with_nearest:
            region_pts = np.argwhere(seg)
            centroid = region_pts.mean(axis=0)
            dists = np.linalg.norm(hint_points.astype(float) - centroid, axis=1)
            nearest_pt = hint_points[np.argmin(dists)]
            region_color = color_map[nearest_pt[0], nearest_pt[1]]
        else:
            continue

        colored_np[seg] = region_color
        filled_mask[seg] = True
        region_colors.append((seg, region_color))

    if has_hints and not filled_mask.any() and outer_mask.any():
        median_color = np.median(color_map[hint_mask[:,:,0] > 0.05], axis=0)
        colored_np[outer_mask] = median_color
        filled_mask[outer_mask] = True
        region_colors.append((outer_mask, median_color))

    return colored_np, filled_mask[..., None].astype(np.float32), region_colors


def overlay_sketch_on_color(colored_np, sketch_np, line_threshold=0.5):
    result = colored_np.copy()
    line_mask = sketch_np < line_threshold
    result[line_mask] = 0.0
    return result


def visualize_coloring(cn_np, colored_np, filled_mask, color_map, hint_mask, sketch_np=None):
    """Show the region-color assignment stage."""
    n_cols = 5 if sketch_np is not None else 4
    fig, axes = plt.subplots(1, n_cols, figsize=(n_cols*4, 4))

    axes[0].imshow(cn_np/255.0 if cn_np.max()>1 else cn_np)
    axes[0].set_title('ControlNet Output'); axes[0].axis('off')

    hint_vis = np.ones_like(colored_np)
    hint_vis[hint_mask[:,:,0]>0] = color_map[hint_mask[:,:,0]>0]
    axes[1].imshow(hint_vis)
    axes[1].set_title('Color Hints'); axes[1].axis('off')

    axes[2].imshow(filled_mask[:,:,0], cmap='gray')
    axes[2].set_title('Selected Region Mask'); axes[2].axis('off')

    axes[3].imshow(colored_np)
    axes[3].set_title('Colored Regions'); axes[3].axis('off')

    if sketch_np is not None and n_cols == 5:
        final_input = overlay_sketch_on_color(colored_np, sketch_np)
        axes[4].imshow(final_input)
        axes[4].set_title('+ Sketch Lines'); axes[4].axis('off')

    plt.tight_layout()
    plt.show()


# ── Test color assignment ──────────────────────────────────────────────────
gt_paths_list = sorted(Path(CONFIG['gt_dir']).glob('*.*'))
sk_paths_list = sorted(Path(CONFIG['sketch_dir']).glob('*.*'))

gt_pil = Image.open(gt_paths_list[0]).convert('RGB').resize((CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)
gt_np = np.array(gt_pil, dtype=np.float32) / 255.0
sk_pil = Image.open(sk_paths_list[0]).convert('L').resize((CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)
sk_np = np.array(sk_pil, dtype=np.float32) / 255.0

color_map, hint_mask = sample_color_hints(gt_np, n_strokes=CONFIG['n_strokes'])
colored_np, filled_mask, _ = assign_colors_to_masks(test_masks, color_map, hint_mask, test_np, outer_mask=outer_test)

visualize_coloring(test_np, colored_np, filled_mask, color_map, hint_mask, sketch_np=sk_np)
print('Color assignment done.')


## Step 4 — LAB Blend

Take luminance (shading, texture) from ControlNet output.
Take hue/saturation from colored regions.
Result: correct structure + correct color, ready for img2img.

In [ ]:
def lab_blend(structured_np, colored_np, region_mask, blend_sigma=2.0):
    """
    Blend structured image luminance with selected region colors.
    region_mask is the full selected/recolored region mask, not sparse hints.
    """
    struct_lab = rgb2lab(structured_np.clip(0,1))
    color_lab = rgb2lab(colored_np.clip(0,1))
    blended_lab = struct_lab.copy()

    weight = region_mask[:,:,0].astype(float)
    if blend_sigma and blend_sigma > 0:
        weight = gaussian_filter(weight, sigma=blend_sigma)
    weight = np.clip(weight, 0, 1)

    blended_lab[:,:,1] = weight * color_lab[:,:,1] + (1 - weight) * struct_lab[:,:,1]
    blended_lab[:,:,2] = weight * color_lab[:,:,2] + (1 - weight) * struct_lab[:,:,2]

    blended_rgb = lab2rgb(blended_lab).astype(np.float32)
    return np.clip(blended_rgb, 0, 1)


# ── Test LAB blend ─────────────────────────────────────────────────────────
struct_float = test_np.astype(float) / 255.0
blended = lab_blend(struct_float, colored_np, filled_mask)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(struct_float); axes[0].set_title('ControlNet (structure)'); axes[0].axis('off')
axes[1].imshow(colored_np); axes[1].set_title('Colored regions'); axes[1].axis('off')
axes[2].imshow(filled_mask[:,:,0], cmap='gray'); axes[2].set_title('Blend mask'); axes[2].axis('off')
axes[3].imshow(blended); axes[3].set_title('LAB blend'); axes[3].axis('off')
plt.tight_layout()
plt.show()
print('LAB blend done.')


## Step 5 — img2img

Takes LAB-blended image as starting point.
Adds realism, texture, cleans background.
Strength=0.5 — preserves structure and color, adds fabric detail.

In [ ]:
print('Loading ControlNet img2img pipeline...')
controlnet_for_img2img = ControlNetModel.from_pretrained(
    CONFIG['controlnet_model'],
    torch_dtype=torch.float16,
)

if CONFIG.get('controlnet_state_dict'):
    state_dict = torch.load(CONFIG['controlnet_state_dict'], map_location='cpu')
    controlnet_for_img2img.load_state_dict(state_dict)
    del state_dict
    print('Loaded fine-tuned ControlNet state dict for img2img.')

img2img_pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    CONFIG['sd_model'],
    controlnet=controlnet_for_img2img,
    torch_dtype=torch.float16,
    safety_checker=None,
).to(device)
img2img_pipe.scheduler = UniPCMultistepScheduler.from_config(img2img_pipe.scheduler.config)
print('ControlNet img2img pipeline loaded.')


def rgb_to_color_name(rgb_01):
    """Convert RGB [0,1] to a basic color name for optional prompt support."""
    import colorsys
    r, g, b = rgb_01
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    h = h * 360
    if v < 0.2:                       return 'black'
    if v > 0.85 and s < 0.12:         return 'white'
    if s < 0.12:                      return 'grey'
    if h < 15 or h >= 345:            return 'red'
    if 15  <= h < 40:                 return 'orange'
    if 40  <= h < 70:                 return 'yellow'
    if 70  <= h < 155:                return 'green'
    if 155 <= h < 200:                return 'teal'
    if 200 <= h < 260:                return 'blue'
    if 260 <= h < 290:                return 'purple'
    if 290 <= h < 345:                return 'pink'
    return 'colourful'


def extract_dominant_colors(color_map, hint_mask, n_colors=2):
    """Extract dominant colors from hint strokes using KMeans."""
    from sklearn.cluster import KMeans
    mask = hint_mask[:,:,0] > 0.05
    pixels = color_map[mask]
    if len(pixels) < 10:
        return []
    n = min(n_colors, len(pixels))
    km = KMeans(n_clusters=n, n_init=3, random_state=42).fit(pixels)
    counts = np.bincount(km.labels_)
    centers = km.cluster_centers_[np.argsort(-counts)]
    names, seen = [], set()
    for c in centers:
        name = rgb_to_color_name(c)
        if name not in seen:
            names.append(name)
            seen.add(name)
    return names


def build_prompt(color_names, category='clothing garment', include_colors=False):
    base = CONFIG['prompt']
    if not include_colors or not color_names:
        return base.replace('clothing item', category) if category != 'clothing garment' else base
    color_str = ' and '.join(color_names)
    return f'a {color_str} {category}, centered, studio product photo, light grey background, no model, no mannequin, high quality'


@torch.no_grad()
def run_img2img(blended_np, sketch_np, color_map, hint_mask,
                category='clothing garment', strength=0.50,
                guidance_scale=7.0, controlnet_scale=1.5,
                n_steps=25, seed=42, include_color_names=False):
    """
    Run ControlNet img2img using:
      - init image: LAB-blended/recolored ControlNet output
      - control image: original sketch/lineart
    """
    input_pil = Image.fromarray((blended_np * 255).astype(np.uint8)).resize(
        (CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)

    if sketch_np.ndim == 2:
        sketch_rgb = np.stack([sketch_np] * 3, axis=-1)
    else:
        sketch_rgb = sketch_np[:,:,:3]
    control_pil = Image.fromarray((sketch_rgb.clip(0,1) * 255).astype(np.uint8)).resize(
        (CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)

    color_names = extract_dominant_colors(color_map, hint_mask)
    prompt = build_prompt(color_names, category, include_colors=include_color_names)
    print(f'  Prompt: {prompt}')

    result = img2img_pipe(
        prompt=prompt,
        negative_prompt=CONFIG['negative_prompt'],
        image=input_pil,
        control_image=control_pil,
        strength=strength,
        guidance_scale=guidance_scale,
        controlnet_conditioning_scale=controlnet_scale,
        num_inference_steps=n_steps,
        generator=torch.Generator(device).manual_seed(seed),
    ).images[0]

    return result


print('ControlNet img2img functions ready.')


## Step 6 — Full Pipeline Test

Runs everything end-to-end on a few samples.
Shows: Sketch | Color Hints | ControlNet | Blended | Final | GT

In [ ]:
def run_full_pipeline(cn_image_path, gt_path, sketch_path,
                       category='clothing garment', n_test_strengths=1,
                       fill_unhinted_with_nearest=False):
    """
    Full pipeline for one sample:
    1. Load ControlNet image, GT, sketch
    2. Sample color hints from GT
    3. Prompted SAM outer garment mask + automatic internal masks
    4. Assign hint colors to selected masks
    5. LAB blend using selected region mask
    6. ControlNet img2img with original sketch as control image
    7. Visualize all stages
    """
    size = CONFIG['image_size']

    cn_pil = Image.open(cn_image_path).convert('RGB').resize((size, size), Image.BILINEAR)
    gt_pil = Image.open(gt_path).convert('RGB').resize((size, size), Image.BILINEAR)
    sk_pil = Image.open(sketch_path).convert('L').resize((size, size), Image.BILINEAR)

    cn_np = np.array(cn_pil)
    gt_np = np.array(gt_pil, dtype=np.float32) / 255.0
    sk_np = np.array(sk_pil, dtype=np.float32) / 255.0

    color_map, hint_mask = sample_color_hints(
        gt_np,
        n_strokes=CONFIG['n_strokes'],
        min_length=CONFIG['stroke_min_length'],
        max_length=CONFIG['stroke_max_length'],
        min_width=CONFIG['stroke_min_width'],
        max_width=CONFIG['stroke_max_width'],
        img_size=size,
    )
    print(f'  Hint pixels: {int(hint_mask.sum())}')

    outer_mask = prompted_outer_mask(cn_np, sk_np)
    masks = filter_masks_to_outer(segment_image(cn_np), outer_mask)
    print(f'  SAM internal regions: {len(masks)}')

    colored_np, filled_mask, _ = assign_colors_to_masks(
        masks, color_map, hint_mask, cn_np,
        outer_mask=outer_mask,
        fill_unhinted_with_nearest=fill_unhinted_with_nearest,
    )

    cn_float = cn_np.astype(float) / 255.0
    blended = lab_blend(cn_float, colored_np, filled_mask)

    results = []
    for s in [CONFIG['img2img_strength']]:
        print(f'  Running ControlNet img2img strength={s}...')
        out = run_img2img(
            blended, sk_np, color_map, hint_mask,
            category=category,
            strength=s,
            guidance_scale=CONFIG['guidance_scale'],
            controlnet_scale=CONFIG['controlnet_scale'],
            n_steps=CONFIG['num_inference_steps'],
            seed=CONFIG['seed'],
        )
        results.append((s, out))

    hint_vis = np.ones((size, size, 3), dtype=np.float32)
    hint_vis[hint_mask[:,:,0] > 0] = color_map[hint_mask[:,:,0] > 0]

    panels = [
        (sk_np, 'Sketch', 'gray'),
        (hint_vis, 'Color Hints', None),
        (cn_float, 'ControlNet', None),
        (outer_mask, 'SAM Outer', 'gray'),
        (filled_mask[:,:,0], 'Selected Color Mask', 'gray'),
        (colored_np, 'Colored Regions', None),
        (blended, 'LAB Blend', None),
    ]
    for s, out in results:
        panels.append((np.array(out) / 255.0, f'CN img2img s={s}', None))
    panels.append((gt_np, 'Ground Truth', None))

    fig, axes = plt.subplots(1, len(panels), figsize=(len(panels) * 3.2, 4))
    for i, (img, title, cmap) in enumerate(panels):
        axes[i].imshow(img.clip(0,1) if hasattr(img, 'clip') else img, cmap=cmap)
        axes[i].set_title(title, fontsize=9)
        axes[i].axis('off')

    plt.suptitle(Path(cn_image_path).stem, fontsize=11)
    plt.tight_layout()

    out_path = os.path.join(CONFIG['output_dir'], f'{Path(cn_image_path).stem}_pipeline.png')
    plt.savefig(out_path, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'  Saved: {out_path}')

    return results[0][1]


print('Pipeline function ready.')


In [ ]:
# ── Run on N samples ────────────────────────────────────────────────────────
N_SAMPLES = 5

gt_paths_all = sorted(Path(CONFIG['gt_dir']).glob('*.*'))
sk_paths_all = sorted(Path(CONFIG['sketch_dir']).glob('*.*'))
cn_paths_all = sorted(glob.glob(os.path.join(CONFIG['controlnet_images_dir'], '*.png')))

if len(cn_paths_all) == 0:
    raise FileNotFoundError(f"No ControlNet images found in {CONFIG['controlnet_images_dir']}")

n_available = min(N_SAMPLES, len(gt_paths_all), len(sk_paths_all), len(cn_paths_all))
indices = random.sample(range(min(len(gt_paths_all), len(sk_paths_all), len(cn_paths_all))), n_available)

print(f'Running pipeline on {n_available} samples...')
for i, idx in enumerate(indices):
    print(f'\nSample {i+1}/{n_available} (index {idx})')
    cn_path = cn_paths_all[idx]
    gt_path = str(gt_paths_all[idx])
    sk_path = str(sk_paths_all[idx])

    try:
        run_full_pipeline(
            cn_image_path=cn_path,
            gt_path=gt_path,
            sketch_path=sk_path,
            category='clothing garment',
            fill_unhinted_with_nearest=False,
        )
    except Exception as e:
        print(f'  Error on sample {idx}: {e}')
        import traceback; traceback.print_exc()
        continue

print('\nAll samples done.')


## Step 7 — Strength Comparison

Test different img2img strength values on one sample to find the best tradeoff
between structure preservation and realism added.

In [ ]:
# Pick one sample for strength comparison
if 'indices' not in globals() or len(indices) == 0:
    indices = [0]

test_idx = indices[0]
cn_path = cn_paths_all[test_idx]
gt_path = str(gt_paths_all[test_idx])
sk_path = str(sk_paths_all[test_idx])
size = CONFIG['image_size']

cn_pil = Image.open(cn_path).convert('RGB').resize((size, size), Image.BILINEAR)
gt_pil = Image.open(gt_path).convert('RGB').resize((size, size), Image.BILINEAR)
sk_pil = Image.open(sk_path).convert('L').resize((size, size), Image.BILINEAR)
cn_np = np.array(cn_pil)
gt_np = np.array(gt_pil, dtype=np.float32) / 255.0
sk_np = np.array(sk_pil, dtype=np.float32) / 255.0

color_map, hint_mask = sample_color_hints(gt_np, n_strokes=CONFIG['n_strokes'])
outer_mask = prompted_outer_mask(cn_np, sk_np)
masks = filter_masks_to_outer(segment_image(cn_np), outer_mask)
colored_np, filled_mask, _ = assign_colors_to_masks(masks, color_map, hint_mask, cn_np, outer_mask=outer_mask)
blended = lab_blend(cn_np.astype(float)/255.0, colored_np, filled_mask)

strengths = [0.35, 0.45, 0.55, 0.65]
outputs = []
for s in strengths:
    print(f'Testing strength={s}...')
    out = run_img2img(
        blended, sk_np, color_map, hint_mask,
        strength=s,
        guidance_scale=CONFIG['guidance_scale'],
        controlnet_scale=CONFIG['controlnet_scale'],
        n_steps=CONFIG['num_inference_steps'],
        seed=CONFIG['seed'],
    )
    outputs.append((s, out))

fig, axes = plt.subplots(1, len(strengths)+3, figsize=((len(strengths)+3)*4, 4))
axes[0].imshow(blended.clip(0,1)); axes[0].set_title('LAB Blend'); axes[0].axis('off')
axes[1].imshow(sk_np, cmap='gray'); axes[1].set_title('Sketch Control'); axes[1].axis('off')
for i, (s, out) in enumerate(outputs):
    axes[i+2].imshow(np.array(out)/255.0)
    axes[i+2].set_title(f'strength={s}'); axes[i+2].axis('off')
axes[-1].imshow(gt_np); axes[-1].set_title('Ground Truth'); axes[-1].axis('off')
plt.suptitle('ControlNet img2img Strength Comparison', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'strength_comparison.png'), dpi=100)
plt.show()
print('Strength comparison saved.')
